## 01 The State

In [ ]:
from typing import TypedDict, Annotated
import operator

# 1. Define the State
# The state is the data structure that gets passed from node to node.
# Annotated[list, operator.add] means whenever a node returns a list of messages, 
# it APPENDS them to the state instead of overwriting the state.
class GraphState(TypedDict):
    messages: Annotated[list, operator.add]
    errors: int


## 02 Nodes And Edges

In [ ]:
from langgraph.graph import StateGraph, START, END

# 2. Define Nodes (Python Functions)
def generator_node(state: GraphState):
    print("--- GENERATING RESPONSE ---")
    return {"messages": ["Here is the generated answer."], "errors": 0}

def validator_node(state: GraphState):
    print("--- VALIDATING RESPONSE ---")
    # Let's pretend it always fails validation for the sake of the example
    return {"errors": state.get("errors", 0) + 1}

# 3. Define Conditional Edges (Routing Logic)
def route_validation(state: GraphState):
    print("--- ROUTING ---")
    if state.get("errors", 0) > 1:
        print("Too many errors, giving up.")
        return END
    else:
        print("Errors found, looping back to generator.")
        return "generator"

# 4. Build the Graph
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("generator", generator_node)
workflow.add_node("validator", validator_node)

# Add edges
workflow.add_edge(START, "generator")
workflow.add_edge("generator", "validator")

# Add conditional edges
workflow.add_conditional_edges(
    "validator",          # The node that just ran
    route_validation,     # The routing function
)

# 5. Compile the Graph
app = workflow.compile()

# Run it!
print("\n=== EXECUTING GRAPH ===")
result = app.invoke({"messages": ["Initial User Question"], "errors": 0})
print("\n=== FINAL STATE ===")
print(result)
